In [1]:
!pip install tensorflow pandas numpy scikit-learn matplotlib seaborn nltk streamlit


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
import pickle
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as plt

In [3]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to C:\Users\VARA
[nltk_data]     PRASAD\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\VARA
[nltk_data]     PRASAD\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [5]:
df = pd.read_csv("dataset/augmented_chatbot_dataset.csv")

df.head()

,Question,Answer
0,What subject is scheduled on Monday at 9:40 AM...,Open Elective – III
1,What class is at 10:40 AM on Monday for Sectio...,Integrated Circuits and Applications Lab (A1) ...
2,What subject is taught at 11:40 AM on Monday f...,Mini Project – II
3,What period is after lunch on Monday for Secti...,Microprocessors and Microcontrollers (MPMC)
4,What subject is from 2:20 PM to 3:20 PM on Mon...,Extra-Curricular Activities – II


In [6]:
df.dropna(inplace=True)

df.drop_duplicates(inplace=True)

df.columns = ['question', 'answer']

print(df.shape)

(4563, 2)


In [7]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words('english'))

def clean_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    words = text.split()

    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

df['question'] = df['question'].apply(clean_text)

df.head()

[nltk_data] Downloading package stopwords to C:\Users\VARA
[nltk_data]     PRASAD\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\VARA
[nltk_data]     PRASAD\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,question,answer
0,subject scheduled monday 940 section,Open Elective – III
1,class 1040 monday section,Integrated Circuits and Applications Lab (A1) ...
2,subject taught 1140 monday section,Mini Project – II
3,period lunch monday section,Microprocessors and Microcontrollers (MPMC)
4,subject 220 pm 320 pm monday section,Extra-Curricular Activities – II


In [8]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df['label'] = label_encoder.fit_transform(df['answer'])

df.head()

,question,answer,label
0,subject scheduled monday 940 section,Open Elective – III,291
1,class 1040 monday section,Integrated Circuits and Applications Lab (A1) ...,232
2,subject taught 1140 monday section,Mini Project – II,268
3,period lunch monday section,Microprocessors and Microcontrollers (MPMC),267
4,subject 220 pm 320 pm monday section,Extra-Curricular Activities – II,206


In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(

    num_words=10000,

    oov_token="<OOV>"
)

tokenizer.fit_on_texts(df['question'])

X = tokenizer.texts_to_sequences(df['question'])

X = pad_sequences(

    X,

    maxlen=20,

    padding='post'
)

y = df['label']

print(X[0])

print(y[0])

[28 56 35 43  3  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
291


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42
)

print("Training samples:", len(X_train))

print("Testing samples:", len(X_test))

Training samples: 3650
Testing samples: 913


In [11]:
from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Embedding,
    Conv1D,
    GlobalMaxPooling1D,
    Dense,
    Dropout
)

model = Sequential([

    Embedding(

        input_dim=10000,

        output_dim=64
    ),

    Conv1D(

        filters=64,

        kernel_size=3,

        activation='relu'
    ),

    GlobalMaxPooling1D(),

    Dense(

        64,

        activation='relu'
    ),

    Dropout(0.3),

    Dense(

        len(label_encoder.classes_),

        activation='softmax'
    )
])

model.compile(

    loss='sparse_categorical_crossentropy',

    optimizer='adam',

    metrics=['accuracy']
)

model.build(input_shape=(None, X_train.shape[1]))

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 20, 64)              │         640,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d (Conv1D)                      │ (None, 18, 64)              │          12,352 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling1d                 │ (None, 64)                  │               0 │
│ (GlobalMaxPooling1D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │           4,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 384)                 │          24,960 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 681,472 (2.60 MB)

 Trainable params: 681,472 (2.60 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(

    monitor='val_loss',

    patience=5,

    restore_best_weights=True
)

history = model.fit(

    X_train,
    y_train,

    epochs=40,

    batch_size=16,

    validation_data=(X_test, y_test),

    callbacks=[early_stop],

    verbose=1
)

Epoch 1/40
229/229 ━━━━━━━━━━━━━━━━━━━━ 8s 18ms/step - accuracy: 0.0268 - loss: 5.5837 - val_accuracy: 0.0537 - val_loss: 5.0105
Epoch 2/40
229/229 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.1249 - loss: 4.3370 - val_accuracy: 0.2782 - val_loss: 3.6245
Epoch 3/40
229/229 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.3871 - loss: 2.7016 - val_accuracy: 0.6396 - val_loss: 1.8306
Epoch 4/40
229/229 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.6827 - loss: 1.3247 - val_accuracy: 0.9036 - val_loss: 0.7236
Epoch 5/40
229/229 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.8362 - loss: 0.6860 - val_accuracy: 0.9770 - val_loss: 0.2704
Epoch 6/40
229/229 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.9060 - loss: 0.4045 - val_accuracy: 0.9869 - val_loss: 0.1209
Epoch 7/40
229/229 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.9345 - loss: 0.2552 - val_accuracy: 0.9945 - val_loss: 0.0633
Epoch 8/40
229/229 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.9608 - loss: 0.1795 - val_accu

In [13]:
import pickle

# Save model
model.save("chatbot_model.keras")

# Save tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Save label encoder
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

print("Model and files saved successfully")

Model and files saved successfully
